# 04 — Interactive Gradio Demo

## Base vs SFT vs DPO

This notebook provides an interactive demonstration of the post-training pipeline.

The user enters a question, and the demo generates responses from:

- **Base model** — original model
- **SFT model** — after supervised fine-tuning
- **DPO model** — after preference alignment

The goal is to visually demonstrate how model behavior changes across the post-training stages.

**Important:** No training is performed in this notebook. The trained LoRA adapters are loaded from Hugging Face.

In [1]:
# Install compatible dependencies for the Gradio demo.
!pip install -q "pyarrow>=18,<22" "pandas>=2.2,<3"
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q gradio

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 122.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 82.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 85.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 118.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.1/225.1 kB 24.5 MB/s eta 0

In [2]:
import torch
import gradio as gr

from unsloth import FastLanguageModel

print("PyTorch:", torch.__version__)
print("Gradio:", gr.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
PyTorch: 2.11.0+cu128
Gradio: 6.24.0
CUDA available: True
GPU: Tesla T4


## Step 1 — Specify the Models

We use the same base model as the training pipeline.

The SFT and DPO components are LoRA adapters that have already been trained and uploaded to Hugging Face.

This allows the demo to reuse the trained models without retraining.

In [3]:
MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"

# Replace these with your exact Hugging Face adapter repository IDs.
SFT_ADAPTER = "Hiteshwari7/posttraining-tutor-sft-adapter"
DPO_ADAPTER = "Hiteshwari7/postraining-tutor-dpo-adapter"

MAX_SEQ_LENGTH = 2048

SYSTEM_PROMPT = (
    "You are PostTraining Tutor, an assistant that explains LLM training "
    "concepts (Transformers, LoRA, QLoRA, SFT, DPO, RLHF, alignment) "
    "clearly and concisely."
)

print("Model configuration loaded.")

Model configuration loaded.


## Step 2 — Load the Base Model

We load the quantized 4-bit base model using Unsloth.

The same model was used during training and evaluation.

The model itself is kept in memory once, while the SFT and DPO adapters are loaded separately.

In [4]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

print("Base model loaded successfully.")

==((====))==  Unsloth 2026.8.19: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-bnb-4bit as a legacy tokenizer.


Base model loaded successfully.


## Step 3 — Load the SFT and DPO Adapters

The SFT and DPO models are LoRA adapters trained on top of the same base model.

We load both adapters into the same model so that we can switch between:

- Base → no adapter
- SFT → SFT adapter
- DPO → DPO adapter

This avoids loading three complete copies of the 3B model.

In [5]:
# Load the two trained adapters.
model.load_adapter(SFT_ADAPTER, adapter_name="sft")
model.load_adapter(DPO_ADAPTER, adapter_name="dpo")

FastLanguageModel.for_inference(model)

print("SFT and DPO adapters loaded successfully.")
print("Available adapters:", model.peft_config.keys())

adapter_config.json:   0%|          | 0.00/1.31k [00:00<?, ?B/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 97.3MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

adapter_config.json:   0%|          | 0.00/1.31k [00:00<?, ?B/s]

...

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 97.3MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

LlamaForCausalLM LOAD REPORT from: Hiteshwari7/postraining-tutor-dpo-adapter
Key                                                      | Status  | 
---------------------------------------------------------+---------+-
model.layers.{0...27}.self_attn.v_proj.lora_A.sft.weight | MISSING | 
model.layers.{0...27}.self_attn.k_proj.lora_A.sft.weight | MISSING | 
model.layers.{0...27}.mlp.gate_proj.lora_B.sft.weight    | MISSING | 
model.layers.{0...27}.self_attn.o_proj.lora_B.sft.weight | MISSING | 
model.layers.{0...27}.self_attn.k_proj.lora_B.sft.weight | MISSING | 
model.layers.{0...27}.self_attn.q_proj.lora_A.sft.weight | MISSING | 
model.layers.{0...27}.mlp.up_proj.lora_A.sft.weight      | MISSING | 
model.layers.{0...27}.mlp.up_proj.lora_B.sft.weight      | MISSING | 
model.layers.{0...27}.self_attn.v_proj.lora_B.sft.weight | MISSING | 
model.layers.{0...27}.self_attn.o_proj.lora_A.sft.weight | MISSING | 
model.layers.{0...27}.self_attn.q_proj.lora_B.sft.weight | MISSING | 
model.layers.

SFT and DPO adapters loaded successfully.
Available adapters: dict_keys(['sft', 'dpo'])


## Step 4 — Create the Generation Function

This function receives:

- a user question
- the selected model stage

It then generates an answer using the same system prompt and generation settings.

The three possible stages are:

**Base / SFT / DPO**

In [6]:
def generate_answer(question, model_stage):
    if not question.strip():
        return "Please enter a question."

    # Select the appropriate model stage.
    if model_stage == "Base":
        model.disable_adapters()

    elif model_stage == "SFT":
        model.enable_adapters()
        model.set_adapter("sft")

    elif model_stage == "DPO":
        model.enable_adapters()
        model.set_adapter("dpo")

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=200,
            do_sample=False,
        )

    answer = tokenizer.decode(
        outputs[0][inputs.shape[1]:],
        skip_special_tokens=True,
    )

    return answer.strip()

## Step 5 — Test Base, SFT, and DPO

Before launching Gradio, we test the generation function on one question.

This confirms that all three model stages can generate responses successfully.

In [7]:
test_question = "What is LoRA?"

print("BASE ")
print(generate_answer(test_question, "Base"))

print("\nSFT")
print(generate_answer(test_question, "SFT"))

print("\nDPO")
print(generate_answer(test_question, "DPO"))

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


BASE 


Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


LoRA stands for Low-Rank Adaptation. It's a technique used in large language model (LLM) training to adapt a pre-trained model to a specific task or dataset.

In traditional LLM training, a model is trained on a large corpus of text data, which can be computationally expensive and time-consuming. LoRA addresses this issue by adapting the model's weights to a lower-rank representation, which is more efficient to compute and store.

Here's how LoRA works:

1. **Pre-training**: A large language model is pre-trained on a massive corpus of text data.
2. **Adaptation**: The pre-trained model is then adapted to a specific task or dataset using a lower-rank matrix (typically 3-4 times lower in dimensionality) that captures the task-specific patterns.
3. **Weight adaptation**: The weights of the pre-trained model are adapted to the lower-rank matrix, which reduces the computational complexity and memory requirements.

LoRA has several

SFT


Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


LoRA (Low-Rank Adaptation) is a technique that adapts a pre-trained model to a new task by modifying a small portion of its weights. This reduces memory usage and computation requirements.

DPO
LoRA (Low-Rank Adaptation) is a method that adapts a pre-trained model to a new task using a smaller adaptation matrix. This reduces memory usage and computation requirements.


## Step 6 — Build the Interactive Gradio Demo

The final interface allows the user to enter one question and receive three answers simultaneously.

This makes the progression:

**Base → SFT → DPO**

easy to demonstrate during the presentation.

In [8]:
def compare_models(question):
    if not question.strip():
        return "Please enter a question.", "Please enter a question.", "Please enter a question."

    base = generate_answer(question, "Base")
    sft = generate_answer(question, "SFT")
    dpo = generate_answer(question, "DPO")

    return base, sft, dpo


demo = gr.Interface(
    fn=compare_models,
    inputs=gr.Textbox(
        label="Enter your question",
        placeholder="Ask a question about LLMs, LoRA, SFT, DPO, RLHF, etc.",
        lines=3,
    ),
    outputs=[
        gr.Textbox(label="Base Model"),
        gr.Textbox(label="SFT Model"),
        gr.Textbox(label="DPO Model"),
    ],
    title="LLM Post-Training Lab",
    description=(
        "Compare responses from the Base, SFT, and DPO stages "
        "of the post-training pipeline."
    ),
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7fb4cf95c1f77b3a69.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
